# CortePro — testar remoção de música

Use pelo celular, com sua conta Google. Execute os blocos **1 a 4**, tocando no botão ▶ de cada bloco e aguardando terminar.

**Primeiro teste:** use um vídeo de 20 a 60 segundos com fala e música. O processo tenta preservar as vozes, mas pode deixar canto, música ou artefatos. Confira o resultado antes de usar.

**Opcional, mais rápido:** em **Ambiente de execução → Alterar tipo de ambiente de execução**, selecione GPU T4 se estiver disponível gratuitamente. Sem GPU, funciona em CPU e pode demorar mais. Não é necessário assinar o Colab Pro. Recursos gratuitos têm limites e podem estar indisponíveis.

Este notebook é um teste manual independente. Ele não altera o baixador nem conecta o Colab como servidor do CortePro. O vídeo é enviado ao ambiente temporário do Google Colab; nenhum Google Drive é montado. Baixe o resultado antes de encerrar a sessão.


In [ ]:
# @title 1. Preparar o removedor (primeira execução pode demorar)
import os, sys, subprocess, shutil
from pathlib import Path

if not (3, 10) <= sys.version_info[:2] <= (3, 12):
    raise RuntimeError("Este notebook usa Python 3.10 a 3.12. Selecione um ambiente Colab com Python 3.12.")

BASE = Path('/content/cortepro_teste')
BASE.mkdir(exist_ok=True)
PYTHON = BASE / 'ambiente/bin/python'
READY = BASE / 'instalado_v1'
if not PYTHON.exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'virtualenv==20.28.0'], check=True)
    subprocess.run([sys.executable, '-m', 'virtualenv', str(BASE / 'ambiente')], check=True)
if not READY.exists():
    print('Instalando o modelo e suas dependências em um ambiente separado. Aguarde...')
    subprocess.run([str(PYTHON), '-m', 'pip', 'install', '--quiet', 'torch==2.5.1',
                    'torchaudio==2.5.1', '--index-url', 'https://download.pytorch.org/whl/cu121'], check=True)
    subprocess.run([str(PYTHON), '-m', 'pip', 'install', '--quiet', 'demucs==4.0.1', 'numpy==1.26.4'], check=True)
    READY.touch()
if not shutil.which('ffmpeg'):
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'ffmpeg'], check=True)
probe = subprocess.run([str(PYTHON), '-c', 'import torch; print("cuda" if torch.cuda.is_available() else "cpu")'],
                       capture_output=True, text=True, check=True)
DEVICE = probe.stdout.strip().splitlines()[-1]
os.environ['MUSIC_DEVICE'] = DEVICE
print('Pronto! Processamento em ' + ('GPU.' if DEVICE == 'cuda' else 'CPU (mais lento).'))
print('Agora execute o bloco 2.')


In [ ]:
# @title 2. Selecionar um vídeo do celular
from google.colab import files
import tempfile

if 'PYTHON' not in globals() or not PYTHON.exists():
    raise RuntimeError('Execute o bloco 1 primeiro.')
# A variável de resultado é limpa antes de qualquer novo envio.
RESULTADO = None
if 'PASTA_VIDEO' in globals():
    shutil.rmtree(PASTA_VIDEO, ignore_errors=True)
PASTA_VIDEO = Path(tempfile.mkdtemp(prefix='video_', dir=BASE))
print('Toque em Escolher arquivos. Selecione apenas UM vídeo, até 200 MB e 10 minutos.')
print('No celular, comece com 20 a 60 segundos para testar a qualidade.')
anterior = Path.cwd()
os.chdir(PASTA_VIDEO)
try:
    enviados = files.upload()
finally:
    os.chdir(anterior)
try:
    if len(enviados) != 1:
        raise ValueError('Selecione apenas um vídeo por vez. Execute este bloco novamente.')
    nome, dados = next(iter(enviados.items()))
    if not 0 < len(dados) <= 200 * 1024 * 1024:
        raise ValueError('O vídeo deve ter entre 1 byte e 200 MB. Envie um arquivo menor.')
    # Escreve a partir dos bytes enviados, sem usar o nome original como caminho.
    entrada = PASTA_VIDEO / 'input.video'
    entrada.write_bytes(dados)
    for arquivo in PASTA_VIDEO.iterdir():
        if arquivo != entrada and arquivo.is_file():
            arquivo.unlink()
    print('Vídeo recebido! Agora execute o bloco 3.')
except Exception:
    shutil.rmtree(PASTA_VIDEO, ignore_errors=True)
    raise
finally:
    del enviados
    if 'dados' in globals(): del dados


In [ ]:
# @title 3. Separar a música e gerar o vídeo
if 'PASTA_VIDEO' not in globals() or not (PASTA_VIDEO / 'input.video').exists():
    raise RuntimeError('Execute o bloco 2 e envie um vídeo primeiro.')
RESULTADO = None
CODIGO_PROCESSADOR = '"""Real source separation; never falls back to muting or filtering the full track."""\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport subprocess\nimport sys\n\n\nclass ProcessingError(Exception):\n    pass\n\n\ndef run(command, timeout=180):\n    return subprocess.run(command, check=True, capture_output=True, timeout=timeout)\n\n\ndef process_video(directory, progress):\n    directory = Path(directory)\n    source = directory / \'input.video\'\n    probe = run([\'ffprobe\', \'-v\', \'error\', \'-protocol_whitelist\', \'file,pipe\', \'-show_streams\', \'-show_format\', \'-of\', \'json\', str(source)], timeout=30)\n    info = json.loads(probe.stdout)\n    streams = info.get(\'streams\', [])\n    if not any(s[\'codec_type\'] == \'video\' for s in streams):\n        raise ProcessingError(\'O arquivo não contém vídeo.\')\n    if not any(s[\'codec_type\'] == \'audio\' for s in streams):\n        raise ProcessingError(\'Este vídeo não possui áudio para tratar.\')\n    try:\n        duration = float(info[\'format\'][\'duration\'])\n    except (KeyError, TypeError, ValueError):\n        raise ProcessingError(\'Não foi possível identificar a duração do vídeo.\')\n    if not math.isfinite(duration) or not 0 < duration <= 600:\n        raise ProcessingError(\'Selecione um vídeo com até 10 minutos.\')\n    progress(\'Extraindo o áudio do vídeo...\')\n    audio = directory / \'audio.wav\'\n    run([\'ffmpeg\', \'-nostdin\', \'-v\', \'error\', \'-y\', \'-protocol_whitelist\', \'file,pipe\', \'-i\', str(source), \'-map\', \'0:a:0\',\n         \'-vn\', \'-ac\', \'2\', \'-ar\', \'44100\', str(audio)], timeout=120)\n    progress(\'Separando a voz da música... isso pode levar vários minutos.\')\n    run([sys.executable, \'-m\', \'demucs\', \'-n\', \'htdemucs\', \'-d\', os.getenv(\'MUSIC_DEVICE\', \'cpu\'),\n         \'--two-stems\', \'vocals\', \'--shifts\', \'1\', \'--segment\', \'7\', \'-j\', \'1\',\n         \'-o\', str(directory / \'stems\'), str(audio)], timeout=1800)\n    vocals = directory / \'stems\' / \'htdemucs\' / \'audio\' / \'vocals.wav\'\n    if not vocals.is_file() or vocals.stat().st_size <= 44:\n        raise ProcessingError(\'O modelo não conseguiu gerar a faixa de voz.\')\n    progress(\'Montando o vídeo com o áudio tratado...\')\n    target = directory / \'voz.mp4\'\n    command = [\'ffmpeg\', \'-nostdin\', \'-v\', \'error\', \'-y\', \'-protocol_whitelist\', \'file,pipe\', \'-i\', str(source), \'-i\', str(vocals),\n               \'-map\', \'0:v:0\', \'-map\', \'1:a:0\', \'-c:v\', \'copy\', \'-c:a\', \'aac\',\n               \'-b:a\', \'192k\', \'-af\', \'apad\', \'-t\', str(duration), \'-movflags\', \'+faststart\', str(target)]\n    try:\n        run(command, timeout=180)\n    except subprocess.CalledProcessError:\n        # Non-MP4 video codecs must be transcoded for Android/browser playback.\n        index = command.index(\'copy\')\n        command[index:index + 1] = [\'libx264\', \'-preset\', \'veryfast\', \'-crf\', \'20\', \'-pix_fmt\', \'yuv420p\']\n        run(command, timeout=180)\n    if not target.is_file() or not target.stat().st_size:\n        raise ProcessingError(\'Não foi possível montar o vídeo tratado.\')\n    # Keep only the deliverable; intermediate audio is deleted even on success.\n    return target\n'
PROCESSADOR = BASE / 'processar.py'
PROCESSADOR.write_text(CODIGO_PROCESSADOR + "\nif __name__ == '__main__':\n    process_video(sys.argv[1], lambda texto: print(texto, flush=True))\n", encoding='utf-8')
print('Iniciando. O primeiro uso também baixa o modelo. Mantenha esta página aberta.')
try:
    subprocess.run([str(PYTHON), str(PROCESSADOR), str(PASTA_VIDEO)], check=True,
                   timeout=2400, env={**os.environ, 'OMP_NUM_THREADS': '2', 'MKL_NUM_THREADS': '2'})
    candidato = PASTA_VIDEO / 'voz.mp4'
    if not candidato.exists() or candidato.stat().st_size == 0:
        raise RuntimeError('Não foi possível gerar o vídeo tratado.')
    RESULTADO = candidato
    for arquivo in PASTA_VIDEO.iterdir():
        if arquivo != RESULTADO:
            if arquivo.is_dir(): shutil.rmtree(arquivo)
            else: arquivo.unlink()
    print('Pronto! Execute o bloco 4 para baixar. Confira se as falas ficaram boas.')
except (subprocess.CalledProcessError, subprocess.TimeoutExpired):
    print('O processamento falhou. Leia a mensagem acima; tente um vídeo mais curto ou selecione GPU e reinicie pelo bloco 1.')
    raise


In [ ]:
# @title 4. Baixar o vídeo com o áudio tratado
from google.colab import files
from IPython.display import display, Video, FileLink

if 'RESULTADO' not in globals() or RESULTADO is None or not RESULTADO.exists():
    raise RuntimeError('Execute os blocos anteriores até aparecer Pronto no bloco 3.')
print('Baixando o arquivo voz.mp4. Confira a pasta Downloads do celular.')
files.download(str(RESULTADO))
# Evita incorporar vídeos grandes no notebook; clipes pequenos podem ser ouvidos aqui.
if RESULTADO.stat().st_size < 20 * 1024 * 1024:
    display(Video(str(RESULTADO), embed=True, html_attributes='controls playsinline style="max-width:100%"'))
print('Para outro vídeo, repita a partir do bloco 2. Isso exclui o resultado anterior do ambiente.')


## Se algo não funcionar

- **Sem GPU gratuita:** teste um clipe curto em CPU ou tente novamente mais tarde.
- **Envio ou download não abre:** abra o link no Chrome e permita os downloads; se necessário, use “Versão para computador” no menu do navegador.
- **Sessão desconectada:** os arquivos podem ser perdidos. Reconecte e execute os blocos novamente.
- **Música ou canto continuam:** o modelo separa vozes de instrumentos; não distingue perfeitamente fala e canto. Não há garantia de remoção total.
- **Somente áudio / sem fala no resultado:** não use o arquivo como substituto do original. Guarde o vídeo original e avalie outro trecho.

Ao terminar, você pode excluir o ambiente em **Ambiente de execução → Desconectar e excluir ambiente de execução**.

Referências: [Demucs](https://github.com/facebookresearch/demucs), [limites do Colab](https://research.google.com/colaboratory/intl/pt-BR/faq.html).
